In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotnine as gg
import numpy as np
import scanpy as sc
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV
from scvi.model import JaxSCVI
from tqdm import tqdm
import scipy.stats as st

from essential.utils import load_kegg_pathways

## load data

In [ ]:
tf_annotations = pd.read_csv(
    "/workspace/data/250516_TF_perturbseq/TF_pathway_annotation_gemini3.csv"
)
tf_annotations["pathway"].value_counts()

In [ ]:
kegg_pathways = load_kegg_pathways()
kegg_pathways.head()

In [ ]:
fitness_data_spacer = pd.read_csv("/workspace/data/calvo2020_dcas9fitness/Supp_data2_log2FC.csv")
display(fitness_data_spacer.head())

In [ ]:
gene_name = "rrlH"
display(fitness_data_spacer.query("gene == @gene_name"))

In [ ]:
adata = sc.read_h5ad("/workspace/data/250516_TF_perturbseq/250516_TF_perturbseq.annotated.h5ad")
random_shuffling = np.random.permutation(np.arange(adata.n_obs))
adata = adata[random_shuffling].copy()
consensus_targets = adata.obs["consensus_target"].unique()

experiment_metadata = pd.read_csv(
    "/workspace/data/250516_TF_perturbseq/library_info.csv"
).set_index("name")
adata.obs = adata.obs.merge(
    experiment_metadata, how="left", left_on="consensus_spacer", right_index=True
)

spacers = adata.obs["spacer"].unique()
fitness_metadata = fitness_data_spacer.set_index("Unnamed: 0").reindex(spacers).fillna(0.0)
adata.obs = adata.obs.merge(fitness_metadata, how="left", left_on="spacer", right_index=True)

In [ ]:
adata_aerobic = adata[adata.obs["consolidated_cluster"] == "aerobic"].copy()

In [ ]:
# JaxSCVI.setup_anndata(adata_aerobic, layer="counts", batch_key="rt_bc")
# model = JaxSCVI(adata_aerobic)
# model.train()

# adata_aerobic.obsm["X_scVI"] = model.get_latent_representation()

# exploration

In [ ]:
TIME_POINT = "T4"

In [ ]:
adata_summary = []
for target in adata_aerobic.obs["spacer"].unique():
    adata_target = adata_aerobic[adata_aerobic.obs["spacer"] == target].copy()
    n_cells = adata_target.shape[0]
    avg_library_size = adata_target.layers["counts"].sum(1).mean()
    min_library_size = adata_target.layers["counts"].sum(1).min()
    max_library_size = adata_target.layers["counts"].sum(1).max()
    adata_summary.append(
        {
            "target": target,
            "n_cells": n_cells,
            "avg_library_size": avg_library_size,
            "min_library_size": min_library_size,
            "max_library_size": max_library_size,
        }
    )
adata_summary = pd.DataFrame(adata_summary).merge(
    fitness_metadata, left_on="target", right_index=True
)

In [ ]:
adata_summary["T4"].hist(bins=50)
plt.show()

In [ ]:
(
    gg.ggplot(adata_summary)
    + gg.geom_point(gg.aes(x="T4", y="n_cells"))
    + gg.theme_minimal()
    + gg.labs(x="fitness (T4)", y="# of cells")
)

In [ ]:
(
    gg.ggplot(adata_summary)
    + gg.geom_point(gg.aes(x="T4", y="min_library_size"))
    + gg.theme_minimal()
    + gg.labs(x="fitness (T4)", y="min. library size")
)

In [ ]:
(
    gg.ggplot(adata_summary)
    + gg.geom_point(gg.aes(x="T4", y="avg_library_size"))
    + gg.theme_minimal()
    + gg.labs(x="fitness (T4)", y="avg. library size")
)

In [ ]:
adata_aerobic.X = adata_aerobic.layers["counts"]
sc.pp.normalize_total(adata_aerobic)
sc.pp.log1p(adata_aerobic)
sc.tl.pca(adata_aerobic, n_comps=50)
sc.pp.neighbors(adata_aerobic)
sc.tl.umap(adata_aerobic, min_dist=0.5)

In [ ]:
sc.pl.umap(adata_aerobic, color=["T2", "T4", "rt_bc"], cmap="bwr", vmin=-1, vmax=1)

In [ ]:
adata_aerobic.obs["rt_bc"].value_counts()

In [ ]:
adata_aerobic_batch = adata_aerobic[adata_aerobic.obs["rt_bc"] == "CGAAAC"].copy()
sc.pp.pca(adata_aerobic_batch, n_comps=10)
sc.pp.neighbors(adata_aerobic_batch, use_rep="X_pca")
sc.tl.umap(adata_aerobic_batch)
sc.pl.umap(adata_aerobic_batch, color=["T2", "T4", TIME_POINT], cmap="bwr", vmin=-1, vmax=1)

In [ ]:
sc.pp.neighbors(adata_aerobic, use_rep="X_scvi_rtbc_corrected")
sc.tl.umap(adata_aerobic, min_dist=0.5)
sc.pl.umap(adata_aerobic, color=["T2", "T4"], cmap="bwr", vmin=-1, vmax=1)

In [ ]:
lr_model = LinearRegression()
lr_model.fit(adata_aerobic.X, adata_aerobic.obs[TIME_POINT])
ypred_lr = lr_model.predict(adata_aerobic.X)

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(6, 5))
plt.scatter(adata_aerobic.obs[TIME_POINT], ypred_lr, s=0.5)
plt.xlabel(TIME_POINT)
plt.ylabel("predicted fitness (all data)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].scatter(adata_aerobic.obs[TIME_POINT], ypred_lr)
axes[0].set_xlabel(TIME_POINT)
axes[0].set_ylabel("predicted fitness (all data)")

axes[1].scatter(adata_aerobic.layers["counts"].sum(1).A1, ypred_lr)
axes[1].set_xlabel("total counts")
axes[1].set_ylabel("predicted fitness (all data)")
axes[1].set_xscale("log")
plt.show()

# Distance to control vs fitness

In [ ]:
control_spacers = experiment_metadata[experiment_metadata.index.str.startswith("Control")][
    "spacer"
].values

In [ ]:
zs = adata_aerobic.obsm["X_scvi_no_correction"]
z0s = adata_aerobic[adata_aerobic.obs["spacer"].isin(control_spacers)].obsm["X_scvi_no_correction"]

# zs = adata_aerobic.X
# z0s = adata_aerobic[adata_aerobic.obs["spacer"].isin(control_spacers)].X

from sklearn.neighbors import NearestNeighbors

# nbrs = NearestNeighbors(n_neighbors=10, metric="cosine").fit(z0s)
nbrs = NearestNeighbors(n_neighbors=10, metric="euclidean").fit(z0s)
distances, indices = nbrs.kneighbors(zs)
adata_aerobic.obs["dist_to_control"] = distances[:, 0]

In [ ]:
summary_df.dropna()

In [ ]:
summary_df = (
    adata_aerobic.obs.groupby(["spacer", "gene"])[["dist_to_control", TIME_POINT]]
    .median()
    .reset_index()
    .loc[lambda x: ~x["spacer"].isin(control_spacers)]
    .dropna()
)

# spearman_ = st.spearmanr(summary_df[TIME_POINT], summary_df["dist_to_control"])[0]
# pearson_ = st.pearsonr(summary_df[TIME_POINT], summary_df["dist_to_control"])[0]

(
    gg.ggplot(summary_df, gg.aes(x=TIME_POINT, y="dist_to_control"))
    # + gg.geom_point(size=0.5)
    + gg.geom_text(gg.aes(label="gene"), size=7)
    + gg.theme_minimal()
    + gg.labs(x=f"fitness ({TIME_POINT})", y="distance to NN in control (using scVI embeddings)")
    # + gg.ggtitle(f"Spearman r: {spearman_:.2f}, Pearson r: {pearson_:.2f}")
)

In [ ]:
# adata_aerobic.X = adata_aerobic.layers["counts"]
# sc.pp.normalize_total(adata_aerobic)
# sc.pp.log1p(adata_aerobic)
# sc.pp.highly_variable_genes(adata_aerobic, n_top_genes=2000, batch_key="rt_bc")
# adata_aerobic = adata_aerobic[:, adata_aerobic.var["highly_variable"]].copy()

Xs = []
for consensus_spacer in adata_aerobic.obs["spacer"].unique():
    Xs.append(
        adata_aerobic[adata_aerobic.obs["spacer"] == consensus_spacer].layers["counts"].sum(0)
    )
Xs = np.array(Xs).squeeze()

adata_aerobic_pseudobulk = sc.AnnData(
    X=Xs,
    var=adata_aerobic.var,
    obs=pd.DataFrame({"spacer": adata_aerobic.obs["spacer"].unique()}),
)

sc.pp.normalize_total(adata_aerobic_pseudobulk)
sc.pp.log1p(adata_aerobic_pseudobulk)

In [ ]:
from sklearn.metrics.pairwise import cosine_distances

In [ ]:
X_ctrl = adata_aerobic_pseudobulk[adata_aerobic_pseudobulk.obs["spacer"].isin(control_spacers)].X

cosine_dists = cosine_distances(X_ctrl, adata_aerobic_pseudobulk.X).min(axis=0)

plot_df = pd.DataFrame(
    {
        "cosine_dist_to_control": cosine_dists.flatten(),
        "spacer": adata_aerobic_pseudobulk.obs["spacer"],
    }
).merge(fitness_metadata, left_on="spacer", right_index=True)

In [ ]:
spearman_ = st.spearmanr(plot_df[TIME_POINT], plot_df["cosine_dist_to_control"])[0]
pearson_ = st.pearsonr(plot_df[TIME_POINT], plot_df["cosine_dist_to_control"])[0]

In [ ]:
plot_df = plot_df.loc[lambda x: x["cosine_dist_to_control"] >= 1e-5]

In [ ]:
(
    gg.ggplot(plot_df, gg.aes(y="cosine_dist_to_control", x=TIME_POINT))
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
    # + gg.scale_x_log10()
    + gg.labs(x="cosine distance to control\n (pseudobulked, log-CPM)", y=f"fitness ({TIME_POINT})")
    + gg.ggtitle(f"Pearson r: {pearson_:.2f}, Spearman r: {spearman_:.2f}")
)

# fitness prediction

In [ ]:
TIME_POINT = "T4"

In [ ]:
fitness_metadata_ = (
    fitness_metadata.reset_index()
    .rename({"Unnamed: 0": "spacer"}, axis=1)
    .merge(tf_annotations, left_on="gene", right_on="gene", how="left")
)

In [ ]:
# pd.set_option("display.max_rows", None)
pd.reset_option("all")

In [ ]:
fitness_metadata_.loc[lambda x: ~x["gene"].str.startswith("Control")]["gene"].unique().shape

In [ ]:
fitness_metadata_.groupby("pathway").apply(
    lambda x: pd.Series(
        {"min": x[TIME_POINT].min(), "max": x[TIME_POINT].max(), "size": x.shape[0]}
    )
).sort_values("size", ascending=False).head(20)

In [ ]:
def process_data(adata):
    adata_ = adata.copy()
    adata_.X = adata_.layers["counts"]
    sc.pp.normalize_total(adata_)
    sc.pp.log1p(adata_)
    return adata_


adata_ = process_data(adata_aerobic)

In [ ]:
heldout_batch = adata_aerobic.obs["rt_bc"].value_counts().idxmin()
heldout_batch

In [ ]:
heldout_genes = fitness_metadata_.loc[lambda x: x["pathway"] == "Carbohydrate Metabolism"][
    "spacer"
].values
# heldout_genes = np.random.permutation(consensus_targets)[:30]

# where_train = lambda x: (~x["consensus_target"].isin(heldout_genes)) & (x["rt_bc"] != heldout_batch)
# where_test = lambda x: x["consensus_target"].isin(heldout_genes) & (x["rt_bc"] == heldout_batch)

where_train = lambda x: (~x["spacer"].isin(heldout_genes))
where_test = lambda x: x["spacer"].isin(heldout_genes)

In [ ]:
adata_train = adata_[adata_.obs.loc[where_train].index]
sc.pp.highly_variable_genes(adata_train, n_top_genes=2000, batch_key="rt_bc")

In [ ]:
adata_train_ = adata_train[:, adata_train.var["highly_variable"]].copy()
adata_test = adata_[adata_.obs.loc[where_test].index]
adata_test_ = adata_test[:, adata_train.var["highly_variable"]].copy()

In [ ]:
y_train_ = adata_train_.obs[TIME_POINT].values
# y_train_ = np.clip(y_train_, -1, 1)

# cv_ = GridSearchCV(
#     estimator=Ridge(),
#     param_grid={"alpha": np.logspace(-1, 4, 10)},
#     cv=3,
#     verbose=10,
# )
# cv_.fit(adata_train_.X.toarray(), adata_train_.obs[TIME_POINT].values)

cv_ = LinearRegression()
cv_.fit(adata_train_.X.toarray(), y_train_)

In [ ]:
ypred_train_ = cv_.predict(adata_train_.X.toarray())
plt.scatter(adata_train_.obs[TIME_POINT], ypred_train_)
plt.show()

In [ ]:
ypred_ = cv_.predict(adata_test_.X.toarray())

In [ ]:
TIME_POINT.replace("_median", "_std")

In [ ]:
plot_df = pd.DataFrame(
    {
        "true": adata_test_.obs[TIME_POINT],
        "pred": ypred_,
        "spacer": adata_test_.obs["spacer"],
        "gene": adata_test_.obs["consensus_target"],
    }
).sort_values("true")

gene_order = plot_df["spacer"].unique()
plot_df["spacer"] = pd.Categorical(plot_df["spacer"], categories=gene_order, ordered=True)


(
    gg.ggplot(plot_df, gg.aes(x="spacer"))
    + gg.geom_boxplot(gg.aes(y="pred"))
    + gg.geom_point(gg.aes(y="true"), size=2, color="red")
    + gg.theme_minimal()
    + gg.labs(x="Target Gene", y="Predicted Fitness")
    + gg.scale_x_discrete(limits=gene_order)
)

In [ ]:
aggregates = plot_df.groupby(["spacer", "gene"])[["true", "pred"]].median().reset_index().dropna()
pearson_ = st.pearsonr(aggregates["true"], aggregates["pred"])
spearmanr_ = st.spearmanr(aggregates["true"], aggregates["pred"])

(
    gg.ggplot(aggregates, gg.aes(x="true", y="pred"))
    # + gg.geom_point()
    + gg.geom_text(gg.aes(label="gene"), size=10)
    # + gg.geom_abline(slope=1, intercept=0, color="red")
    + gg.theme_minimal()
    + gg.labs(
        x=f"GT fitness ({TIME_POINT})",
        y="pred. fitness\n(predicted per capsule; averaged over sgRNAs)",
    )
    + gg.ggtitle(f"Pearson r: {pearson_[0]:.2f}, Spearman r: {spearmanr_[0]:.2f}")
)

In [ ]:
fitness_data_raw.loc[lambda x: x["gene"] == "crp"]